In [8]:
!pip install torchvision opencv-python


extracting static visual features (frame embeddings) using a lightweight CNN like ResNet18 from torchvision

In [9]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
import cv2
import numpy as np

# Load ResNet18 without the classification head
model = models.resnet18(pretrained=True)
model = torch.nn.Sequential(*list(model.children())[:-1])  # Remove final FC layer
model.eval()  # Inference mode


Sequential(
  (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (4): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Con

In [10]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

def extract_video_features(video_path, model, transform, frame_skip=5):
    cap = cv2.VideoCapture(video_path)
    features = []
    frame_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_count % frame_skip == 0:
            try:
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                img = transform(frame_rgb).unsqueeze(0)  # shape: [1, 3, 224, 224]
                with torch.no_grad():
                    feat = model(img).squeeze().numpy()  # shape: [512]
                features.append(feat)
            except:
                continue
        frame_count += 1

    cap.release()
    features = np.array(features)
    return np.mean(features, axis=0)  # Average frame features


In [11]:
from tqdm import tqdm
import os

video_folder = '/content'
X = []
y = []

for file in tqdm(os.listdir(video_folder)):
    if file.endswith('.mkv'):
        label = 1 if 'goal' in file.lower() else 0
        path = os.path.join(video_folder, file)
        feat = extract_video_features(path, model, transform)
        X.append(feat)
        y.append(label)

X = np.array(X)
y = np.array(y)
print(X.shape, y.shape)  # (num_videos, 512), (num_videos,)


100%|██████████| 30/30 [02:20<00:00,  4.68s/it]

(28, 512) (28,)


In [12]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.6666666666666666
              precision    recall  f1-score   support

           0       1.00      0.33      0.50         3
           1       0.60      1.00      0.75         3

    accuracy                           0.67         6
   macro avg       0.80      0.67      0.62         6
weighted avg       0.80      0.67      0.62         6



motion features now: using frame differencing

In [13]:
import cv2
import numpy as np

def extract_motion_features(video_path, max_frames=30):
    cap = cv2.VideoCapture(video_path)
    prev_frame = None
    diffs = []

    count = 0
    while True:
        ret, frame = cap.read()
        if not ret or count >= max_frames:
            break

        frame = cv2.resize(frame, (112, 112))
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        if prev_frame is not None:
            diff = cv2.absdiff(gray, prev_frame)
            diffs.append(np.mean(diff))

        prev_frame = gray
        count += 1

    cap.release()

    diffs = np.array(diffs)
    if len(diffs) == 0:
        return np.zeros(3)

    return np.array([np.mean(diffs), np.std(diffs), np.max(diffs)])


In [14]:
video_files = [f for f in os.listdir('/content') if f.endswith('.mkv')]
X_static = []
X_motion = []
y = []

for file in tqdm(video_files):
    label = 1 if 'goal' in file.lower() else 0
    path = os.path.join('/content', file)

    # Static features
    static_feat = extract_video_features(path, model, transform)  # already done

    # Motion features
    motion_feat = extract_motion_features(path)


    X_static.append(static_feat)
    X_motion.append(motion_feat)
    y.append(label)

# Concatenate
X_combined = [np.concatenate([s, m]) for s, m in zip(X_static, X_motion)]


100%|██████████| 28/28 [02:29<00:00,  5.33s/it]


In [15]:
X_train, X_test, y_train, y_test = train_test_split(X_combined, y, test_size=0.2, stratify=y)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.6666666666666666
              precision    recall  f1-score   support

           0       1.00      0.33      0.50         3
           1       0.60      1.00      0.75         3

    accuracy                           0.67         6
   macro avg       0.80      0.67      0.62         6
weighted avg       0.80      0.67      0.62         6



In [16]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

X_train, X_test, y_train, y_test = train_test_split(X_motion, y, test_size=0.2, stratify=y)


clf_motion = LogisticRegression(max_iter=1000)
clf_motion.fit(X_train, y_train)


y_pred = clf_motion.predict(X_test)
print(" Motion-Only Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


 Motion-Only Accuracy: 0.5
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         3
           1       0.50      1.00      0.67         3

    accuracy                           0.50         6
   macro avg       0.25      0.50      0.33         6
weighted avg       0.25      0.50      0.33         6



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [17]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report


X_train, X_test, y_train, y_test = train_test_split(X_static, y, test_size=0.2, stratify=y)


clf_static = LogisticRegression(max_iter=1000)
clf_static.fit(X_train, y_train)

# Evaluate
y_pred = clf_static.predict(X_test)
print(" Static-Only Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


 Static-Only Accuracy: 0.6666666666666666
              precision    recall  f1-score   support

           0       0.67      0.67      0.67         3
           1       0.67      0.67      0.67         3

    accuracy                           0.67         6
   macro avg       0.67      0.67      0.67         6
weighted avg       0.67      0.67      0.67         6

